In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

np.random.seed(42)

In [2]:
BASE_DIR = Path(
    "paper_implementation"
)

PROFILE_DIR = Path(
    "daily_profiles_24h"
)

MODE3_DIR = (
    BASE_DIR /
    "theft_simulation" /
    "mode_3"
)

MODE3_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
selected_df = pd.read_csv(
    BASE_DIR /
    "selected_150_meters" /
    "selected_150_meter_ids.csv"
)

fraud_df = pd.read_csv(
    BASE_DIR /
    "area_assignments" /
    "fraud_consumers.csv"
)

selected_meters = (
    selected_df["Meter"]
    .astype(str)
    .tolist()
)

fraud_meters = set(
    fraud_df["Meter"]
    .astype(str)
    .tolist()
)

print(
    "Selected Consumers:",
    len(selected_meters)
)

print(
    "Fraud Consumers:",
    len(fraud_meters)
)

Selected Consumers: 150
Fraud Consumers: 27


In [4]:
hour_cols = [
    f"HOUR_{i}"
    for i in range(24)
]

In [5]:
total_modified = 0

log_records = []

for meter in selected_meters:

    df = pd.read_csv(
        PROFILE_DIR /
        f"{meter}.csv"
    )

    # Apply theft only to fraud consumers

    if meter in fraud_meters:

        for row_idx in df.index:

            date = df.loc[
                row_idx,
                "DATE"
            ]

            # Maximum reading of that day

            daily_max = (
                df.loc[
                    row_idx,
                    hour_cols
                ]
                .astype(float)
                .max()
            )

            # Generate ONE gamma for the entire day

            gamma = np.random.uniform(
                0,
                daily_max
            )

            for hour in hour_cols:

                x_it = float(
                    df.loc[
                        row_idx,
                        hour
                    ]
                )

                # Equation (10)

                x_prime = max(
                    x_it - gamma,
                    0
                )

                # Save log

                log_records.append({

                    "Meter":
                    meter,

                    "Date":
                    date,

                    "Hour":
                    hour,

                    "Actual_Reading":
                    x_it,

                    "Daily_Max":
                    daily_max,

                    "Gamma":
                    gamma,

                    "x_prime":
                    x_prime
                })

                # Replace reading

                df.loc[
                    row_idx,
                    hour
                ] = x_prime

                total_modified += 1

    # Save consumer file

    df.to_csv(
        MODE3_DIR /
        f"{meter}.csv",
        index=False
    )

# Save log

log_df = pd.DataFrame(
    log_records
)

log_df.to_csv(
    BASE_DIR /
    "theft_simulation" /
    "mode_3_generation_log.csv",
    index=False
)

print(
    "Modified readings:",
    total_modified
)

print(
    "Log rows:",
    len(log_df)
)

Modified readings: 20088
Log rows: 20088


In [6]:
print(
    "Files Generated:",
    len(
        list(
            MODE3_DIR.glob("*.csv")
        )
    )
)

log_df = pd.read_csv(
    BASE_DIR /
    "theft_simulation" /
    "mode_3_generation_log.csv"
)

print(
    "Log Rows:",
    len(log_df)
)

Files Generated: 150
Log Rows: 20088


In [7]:
normal_meters = [
    m for m in selected_meters
    if m not in fraud_meters
]

unchanged_count = 0

for meter in normal_meters:

    original = pd.read_csv(
        PROFILE_DIR /
        f"{meter}.csv"
    )

    mode3 = pd.read_csv(
        MODE3_DIR /
        f"{meter}.csv"
    )

    same = np.allclose(
        original[hour_cols].values,
        mode3[hour_cols].values
    )

    if same:
        unchanged_count += 1

print(
    "Unchanged Normal Consumers:",
    unchanged_count,
    "/",
    len(normal_meters)
)

Unchanged Normal Consumers: 123 / 123


In [8]:
modified_count = 0

for meter in fraud_meters:

    original = pd.read_csv(
        PROFILE_DIR /
        f"{meter}.csv"
    )

    mode3 = pd.read_csv(
        MODE3_DIR /
        f"{meter}.csv"
    )

    same = np.allclose(
        original[hour_cols].values,
        mode3[hour_cols].values
    )

    if not same:
        modified_count += 1

print(
    "Modified Fraud Consumers:",
    modified_count,
    "/",
    len(fraud_meters)
)

Modified Fraud Consumers: 27 / 27


In [12]:
(
    (
        log_df["Daily_Max"] == 0
    )
    |
    (
        log_df["Gamma"]
        <
        log_df["Daily_Max"]
    )
).all()

np.True_

In [13]:
log_df.head(20)

,Meter,Date,Hour,Actual_Reading,Daily_Max,Gamma,x_prime
0,6270,2018-07-01,HOUR_0,1.2354,5.2638,1.971504,0.000000
1,6270,2018-07-01,HOUR_1,1.1880,5.2638,1.971504,0.000000
2,6270,2018-07-01,HOUR_2,0.8430,5.2638,1.971504,0.000000
3,6270,2018-07-01,HOUR_3,0.5580,5.2638,1.971504,0.000000
4,6270,2018-07-01,HOUR_4,0.5826,5.2638,1.971504,0.000000
5,6270,2018-07-01,HOUR_5,0.5688,5.2638,1.971504,0.000000
6,6270,2018-07-01,HOUR_6,0.5652,5.2638,1.971504,0.000000
7,6270,2018-07-01,HOUR_7,0.5376,5.2638,1.971504,0.000000
8,6270,2018-07-01,HOUR_8,0.8574,5.2638,1.971504,0.000000
9,6270,2018-07-01,HOUR_9,0.6018,5.2638,1.971504,0.000000
